<a href="https://colab.research.google.com/github/minnachirakkal775-bot/WikiScroll_AI_write/blob/main/Mini_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q groq gradio wikipedia-api

In [ ]:
import os
import random
import time
import gradio as gr
import wikipediaapi

from groq import Groq
from google.colab import userdata

In [ ]:
client = Groq(
    api_key=userdata.get("YOUR_API_KEY")
)

In [ ]:
wiki = wikipediaapi.Wikipedia(
    user_agent="DoomScrollBot/1.0",
    language="en"
)

In [ ]:
knowledge_base = []
generated_cards = []
search_history = []
liked_cards = []

current_index = -1
cards_generated = 0
auto_mode = False

In [ ]:
def fetch_wikipedia_pages(topic):
    pages = []

    search_topics = [
        topic,
        topic + " history",
        topic + " culture"
    ]

    for title in search_topics:
        page = wiki.page(title)

        if page.exists():
            pages.append({
                "title": page.title,
                "content": page.text
            })

    return pages

In [ ]:
def create_chunks(text, chunk_size=400):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])

        if len(chunk.split()) > 100:
            chunks.append(chunk)

    return chunks

In [ ]:
def load_topic(topic):
    global knowledge_base, generated_cards, search_history, current_index, cards_generated

    knowledge_base = []
    generated_cards = []
    current_index = -1
    cards_generated = 0

    if topic not in search_history:
        search_history.append(topic)

    pages = fetch_wikipedia_pages(topic)

    for page in pages:
        chunks = create_chunks(page["content"])

        for chunk in chunks:
            knowledge_base.append({
                "title": page["title"],
                "chunk": chunk
            })

    return (
        f"{len(knowledge_base)} chunks loaded",
        "\n".join(search_history),
        0
    )

In [ ]:
def generate_insight():
    if not knowledge_base:
        return "Load a topic first.", ""

    item = random.choice(knowledge_base)

    context = item["chunk"]
    source = item["title"]

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0.8,
        max_tokens=300,
        messages=[
            {
                "role": "system",
                "content": """
You are a Wikipedia-based content generator.

Rules:
1. Use ONLY the supplied passage.
2. Do not use outside knowledge.
3. Generate exactly 5 bullet-point insights.
4. Keep each point short.
5. Return only bullet points.
"""
            },
            {
                "role": "user",
                "content": f"""
Wikipedia Passage:

{context}

Generate 5 insights.
"""
            }
        ]
    )

    return response.choices[0].message.content.strip(), source

In [ ]:
def next_card():
    global cards_generated, current_index, generated_cards

    insight, source = generate_insight()

    generated_cards.append(insight)
    cards_generated += 1
    current_index = len(generated_cards) - 1

    wiki_url = f"https://en.wikipedia.org/wiki/{source.replace(' ', '_')}"

    source_html = f"""
    <p>📌 <b>Source:</b>
    <a href="{wiki_url}" target="_blank" style="color:#38bdf8; text-decoration:none;">
    {source}
    </a>
    </p>
    """

    html = f"""
    <div style="
        background: rgba(255,255,255,0.08);
        backdrop-filter: blur(12px);
        padding: 30px;
        border-radius: 25px;
        max-width: 900px;
        margin: auto;
        color: white;
        box-shadow: 0px 10px 30px rgba(0,0,0,0.4);
    ">

        <h2 style="text-align:center;color:#facc15;">
            📖 Doom Card #{cards_generated}
        </h2>

        <hr style="opacity:0.2;">

        <h3 style="color:#38bdf8;">✨ Insights</h3>

        <div style="font-size:18px;line-height:2;white-space:pre-wrap;">
{insight}
        </div>

        <hr style="opacity:0.2;">

        {source_html}

        <p>📊 <b>Total Cards:</b> {cards_generated}</p>

    </div>
    """

    history_text = "\n\n".join(
        [f"Card {i+1}\n{card}" for i, card in enumerate(generated_cards)]
    )

    return html, cards_generated, history_text

In [ ]:
def like_current_card():
    if generated_cards and current_index >= 0:
        liked_cards.append(generated_cards[current_index])
        return f"❤️ Liked Cards: {len(liked_cards)}"
    return "No card to like"

In [ ]:
def download_cards():
    filename = "cards.txt"

    with open(filename, "w", encoding="utf-8") as f:
        for i, card in enumerate(generated_cards):
            f.write(f"Card {i+1}\n{card}\n\n")

    return filename

In [ ]:
def download_liked():
    filename = "liked_cards.txt"

    with open(filename, "w", encoding="utf-8") as f:
        for i, card in enumerate(liked_cards):
            f.write(f"Liked Card {i+1}\n{card}\n\n")

    return filename

In [ ]:
custom_css = """
body {
    background: linear-gradient(-45deg, #0b001a, #2e1065, #7c3aed, #ec4899);
    background-size: 400% 400%;
    animation: gradientBG 10s ease infinite;
    color: white;
}

@keyframes gradientBG {
    0% {background-position: 0% 50%;}
    50% {background-position: 100% 50%;}
    100% {background-position: 0% 50%;}
}

.gr-button {
    background: #a855f7 !important;
    color: white !important;
    border-radius: 12px !important;
}

.gr-textbox, .gr-box {
    border-radius: 12px !important;
    background: rgba(255,255,255,0.05) !important;
    color: white !important;
    border: 1px solid rgba(255,255,255,0.1) !important;
}
"""

with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as demo:

    gr.Markdown("# 🚀 Doom Scroll AI")
    gr.Markdown("### Endless AI Wikipedia Feed")

    topic_input = gr.Textbox(label="Enter Topic")
    load_btn = gr.Button("Load Topic")

    status_box = gr.Textbox(label="Status")
    card_counter = gr.Number(label="Cards Generated", value=0)
    history_box = gr.Textbox(label="Search History", lines=4)

    card_output = gr.HTML()
    insights_history = gr.Textbox(label="All Insights", lines=12)

    with gr.Row():
        prev_btn = gr.Button("⬅ Previous")
        next_btn = gr.Button("➡ Next Insight")

    with gr.Row():
        like_btn = gr.Button("❤️ Like")
        download_btn = gr.Button("📥 Download All")
        download_liked_btn = gr.Button("💾 Download Liked")

    file_output = gr.File()

    # EVENTS
    load_btn.click(load_topic,
                   inputs=topic_input,
                   outputs=[status_box, history_box, card_counter])

    next_btn.click(next_card,
                   outputs=[card_output, card_counter, insights_history])

    like_btn.click(like_current_card,
                   outputs=status_box)

    download_btn.click(download_cards,
                       outputs=file_output)

    download_liked_btn.click(download_liked,
                             outputs=file_output)

demo.launch(share=True)

/tmp/ipykernel_2636/681028675.py:29: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as demo:
/tmp/ipykernel_2636/681028675.py:29: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://10b375aef9ce59280f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
